# 02 — Modélisation fraude

Objectifs :
- comparer les modèles supervisés sur un échantillon d'entraînement harmonisé ;
- visualiser ROC / PR et la matrice de confusion ;
- confronter split aléatoire et validation temporelle (`step`) ;
- analyser les faux négatifs et quasi-faux positifs.

Le pipeline de production est dans `src/models/train_fraud_model.py`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FRAUD_DIR = ROOT / "models" / "fraud"
sns.set_theme(style="whitegrid")

## 1. Comparaison des modèles (split aléatoire)

In [ ]:
comparison = pd.read_csv(FRAUD_DIR / "fraud_model_comparison.csv", index_col=0)
best = json.loads((FRAUD_DIR / "fraud_best_model.json").read_text(encoding="utf-8"))
comparison[["pr_auc", "recall", "precision", "f1", "threshold", "train_size"]]

In [ ]:
metrics_plot = comparison[["pr_auc", "recall", "precision", "f1"]]
ax = metrics_plot.plot(kind="bar", figsize=(10, 4), rot=0)
ax.set_ylim(0, 1.05)
ax.set_title("Comparaison des modèles — split aléatoire stratifié")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

## 2. Courbes ROC et Precision-Recall (modèle retenu)

In [ ]:
import joblib
from src.models.train_fraud_model import (
    DEFAULT_MAX_TRAIN_ROWS,
    _best_threshold_from_validation,
    _prepare_fraud_dataset,
    _stratified_sample,
)

X, y = _prepare_fraud_dataset()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1765, random_state=42, stratify=y_train_full
)
X_train, y_train = _stratified_sample(X_train, y_train, max_rows=DEFAULT_MAX_TRAIN_ROWS)

model = joblib.load(FRAUD_DIR / "fraud_model.joblib")
test_proba = model.predict_proba(X_test)[:, 1]
threshold = float(comparison.loc[best["best_model"], "threshold"])
test_pred = (test_proba >= threshold).astype(int)

print(f"Modèle retenu : {best['best_model']}")
print(f"Seuil : {threshold:.2f}")
print(f"ROC-AUC test : {roc_auc_score(y_test, test_proba):.4f}")
print(f"PR-AUC test : {average_precision_score(y_test, test_proba):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
RocCurveDisplay.from_predictions(y_test, test_proba, ax=axes[0], name=best["best_model"])
PrecisionRecallDisplay.from_predictions(y_test, test_proba, ax=axes[1], name=best["best_model"])
axes[0].set_title("Courbe ROC")
axes[1].set_title("Courbe Precision-Recall")
plt.tight_layout()
plt.show()

## 3. Matrice de confusion

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, cmap="Blues", ax=ax)
ax.set_title(f"Matrice de confusion — seuil {threshold:.2f}")
plt.tight_layout()
plt.show()

## 4. Validation temporelle (`step`)

In [ ]:
temporal_path = FRAUD_DIR / "fraud_temporal_metrics.json"
if temporal_path.exists():
    temporal = json.loads(temporal_path.read_text(encoding="utf-8"))
    compare_df = pd.DataFrame(
        {
            "split_aleatoire": comparison.loc[best["best_model"], ["pr_auc", "recall", "precision", "f1"]],
            "split_temporel": pd.Series(temporal)[["pr_auc", "recall", "precision", "f1"]],
        }
    )
    display(compare_df)
    compare_df.plot(kind="bar", figsize=(8, 4), rot=0)
    plt.title("Split aléatoire vs split temporel")
    plt.ylabel("Score")
    plt.tight_layout()
    plt.show()
else:
    print("Lancer : python -m src.models.train_fraud_model")

## 5. Analyse des erreurs (FN / quasi-FP)

In [ ]:
error_path = FRAUD_DIR / "fraud_error_analysis.json"
if error_path.exists():
    errors = json.loads(error_path.read_text(encoding="utf-8"))
    print("Résumé :", {k: errors[k] for k in errors if not k.endswith("_cases")})
    fn_df = pd.read_csv(FRAUD_DIR / "fraud_false_negatives.csv")
    fp_df = pd.read_csv(FRAUD_DIR / "fraud_false_positives.csv")
    display(fn_df)
    display(fp_df)
else:
    print("Lancer : python -m src.models.train_fraud_model")

## 6. Périmètre des modèles (énoncé vs implémenté)

## 6bis. Analyse SHAP (modèle retenu)

In [ ]:
from IPython.display import Image, display

shap_fig = ROOT / "reports" / "figures" / "09_fraud_shap_summary.png"
if shap_fig.exists():
    display(Image(filename=str(shap_fig)))
else:
    print("Générer la figure : python -m src.visualization.generate_report_figures")

In [ ]:
scope_path = FRAUD_DIR / "fraud_models_scope.json"
if scope_path.exists():
    pd.DataFrame(json.loads(scope_path.read_text(encoding="utf-8")))
else:
    print("Tableau de périmètre non généré.")

## Interprétation

- XGBoost, Random Forest, LightGBM et la régression logistique utilisent le **train complet** (~734 000 lignes).
- Le **MLP** est entraîné sur **200 000 lignes** (coût calcul du réseau de neurones).
- **XGBoost** est retenu sur la PR-AUC ; le seuil est calibré sur validation avec rappel minimal.
- La validation temporelle vérifie si le modèle tient lorsque les périodes futures (`step`) n'ont pas été vues à l'entraînement.
- Les faux négatifs doivent être analysés manuellement : ils représentent le coût financier résiduel.
- Les quasi-faux positifs (seuil abaissé à 0,50) illustrent la charge opérationnelle si le seuil était plus permissif.

## Reproduction

```bash
python -m src.models.train_fraud_model
python -m src.visualization.generate_report_figures
```